# Práctica: nnU-Net como framework auto-configurable (MONAI `nnUNetV2Runner`)

En la [clase 3](../clase3.qmd) discutimos que nnU-Net no es "otra U-Net", sino un **framework auto-configurable**: analiza el dataset (su *fingerprint*) y deriva automáticamente decisiones de pipeline (spacing, 2D/3D, tamaño de *patch*, normalización, postprocesamiento) que tradicionalmente se ajustan a mano.

En este notebook vamos a **recorrer y visualizar cada paso** de ese pipeline usando la integración oficial de MONAI (`monai.apps.nnunet.nnUNetV2Runner`), sobre el mismo dataset **Task09_Spleen** (Medical Segmentation Decathlon) que usamos en el `Notebook1`.

Los pasos que vamos a ejecutar son:

```{mermaid}
flowchart TB
    D[Dataset MSD] --> C[1. Conversión al formato nnU-Net]
    C --> F[2. Planning + preprocesamiento]
    F --> T[3. Entrenar un único modelo]
    T --> B[4. Encontrar la mejor configuración]
    B --> P[5. Predicción]
    P --> PP[6. Postprocesamiento]
```

::: {.callout-important}
Por tiempo y recursos de GPU, en Colab vamos a entrenar **un solo fold** de la configuración `3d_fullres`, con un *trainer* reducido (`nnUNetTrainer_5epochs`). En un uso real de nnU-Net se entrenan **5 folds** por cada una de varias configuraciones (`2d`, `3d_fullres`, `3d_lowres`, `3d_cascade_fullres`) y luego se comparan/ensamblan. El objetivo acá es **entender el pipeline**, no reproducir un modelo con desempeño de producción.
:::

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ignacio-scarinci/ia-fisica-medica/blob/main/unidad2/notebooks/Notebook6.ipynb)

## Preparación del entorno

Necesitamos `monai` (para el runner y para descargar el dataset) y `nnunetv2` (el paquete real de nnU-Net, que hace todo el trabajo pesado). Recomendado: activar un runtime con **GPU** en Colab (`Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU`).

In [ ]:
!python -c "import monai" || pip install -q "monai-weekly[gdown, nibabel, tqdm]"
!python -c "import nnunetv2" || pip install -q nnunetv2
!python -c "import matplotlib" || pip install -q matplotlib
%matplotlib inline

In [ ]:
import glob
import json
import os
import shutil
import tempfile

import matplotlib.pyplot as plt
import nibabel as nib

from monai.apps import download_and_extract
from monai.apps.nnunet import nnUNetV2Runner
from monai.config import print_config

print_config()

## Descarga del dataset (Task09_Spleen)

Mismo dataset y mismo mecanismo de descarga que en el `Notebook1`: 61 volúmenes de TC con máscara del bazo, del Medical Segmentation Decathlon.

In [ ]:
directory = os.environ.get("MONAI_DATA_DIRECTORY")
if directory is not None:
    os.makedirs(directory, exist_ok=True)
root_dir = tempfile.mkdtemp() if directory is None else directory
print(root_dir)

In [ ]:
resource = "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task09_Spleen.tar"
md5 = "410d4a301da4e5b2f6f86ec3ddba524e"

compressed_file = os.path.join(root_dir, "Task09_Spleen.tar")
msd_data_dir = os.path.join(root_dir, "Task09_Spleen")
if not os.path.exists(msd_data_dir):
    download_and_extract(resource, compressed_file, root_dir, md5)

## Directorios de trabajo de nnU-Net

nnU-Net organiza todo alrededor de **tres carpetas** (equivalentes a variables de entorno `nnUNet_raw`, `nnUNet_preprocessed`, `nnUNet_results`):

- **raw**: el dataset ya convertido al formato que nnU-Net espera.
- **preprocessed**: el resultado del *fingerprint* + preprocesamiento (lo que alimenta al entrenamiento).
- **results**: checkpoints, logs de entrenamiento, y finalmente las predicciones.

`nnUNetV2Runner` puede crear y setear estas tres carpetas por nosotros a partir de un diccionario de configuración (`input_config`).

In [ ]:
work_dir = os.path.join(root_dir, "nnunet_work_dir")

input_config = {
    "modality": "CT",
    "dataset_name_or_id": 9,  # coincide con el "09" de Task09_Spleen
    "nnunet_raw": os.path.join(work_dir, "nnUNet_raw"),
    "nnunet_preprocessed": os.path.join(work_dir, "nnUNet_preprocessed"),
    "nnunet_results": os.path.join(work_dir, "nnUNet_trained_models"),
}

runner = nnUNetV2Runner(
    input_config=input_config,
    trainer_class_name="nnUNetTrainer_5epochs",  # reducido para que corra en Colab; en producción: "nnUNetTrainer"
    work_dir=work_dir,
)

## Paso 1 — Conversión del dataset

El dataset MSD ya viene en un formato muy cercano al de nnU-Net, así que usamos el atajo `convert_msd_dataset` (en vez de armar manualmente un `datalist.json`). Este paso:

1. copia las imágenes/etiquetas a `nnUNet_raw` con la convención de nombres que nnU-Net necesita (sufijo `_0000` por canal);
2. genera el archivo `dataset.json` (modalidades, clases, cantidad de casos).

In [ ]:
runner.convert_msd_dataset(data_dir=msd_data_dir)

### Visualizar el resultado de la conversión

En vez de confiar "a ciegas" en el paso anterior, miramos el árbol de carpetas generado en `nnUNet_raw` y el contenido de `dataset.json`.

In [ ]:
for root, dirs, files in os.walk(runner.nnunet_raw):
    level = root.replace(runner.nnunet_raw, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level >= 1:
        for f in sorted(files)[:3]:
            print(f"{indent}  {f}")
        if len(files) > 3:
            print(f"{indent}  ... ({len(files)} archivos en total)")

In [ ]:
dataset_json_path = os.path.join(runner.nnunet_raw, os.listdir(runner.nnunet_raw)[0], "dataset.json")
with open(dataset_json_path) as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

## Paso 2 — Experiment planning + preprocesamiento

Acá es donde nnU-Net calcula el **dataset fingerprint** ($D \rightarrow F(D)$, ver clase 3): dimensiones, *spacing*, anisotropía, distribución de intensidades. A partir de eso, el *Experiment Planner* decide spacing objetivo, tamaño de *patch*, batch size y qué configuraciones (2D, 3D full-res, 3D low-res) tienen sentido para este dataset — y finalmente preprocesa los volúmenes (resampling, normalización) según ese plan.

::: {.callout-note}
Esto puede tardar varios minutos: nnU-Net recorre los 41 volúmenes de entrenamiento para construir el fingerprint.
:::

In [ ]:
runner.plan_and_process(verify_dataset_integrity=True)

### Inspeccionar el fingerprint y el plan generados

Estos dos archivos JSON son la evidencia concreta de la "auto-configuración": ningún humano eligió a mano el spacing o el tamaño de *patch*, se derivaron de los datos.

In [ ]:
dataset_dir_name = os.listdir(runner.nnunet_preprocessed)[0]
preprocessed_dataset_dir = os.path.join(runner.nnunet_preprocessed, dataset_dir_name)

with open(os.path.join(preprocessed_dataset_dir, "dataset_fingerprint.json")) as f:
    fingerprint = json.load(f)

print("Spacings medianos observados:", fingerprint.get("median_relative_size_after_cropping", "n/d"))
print("Claves principales del fingerprint:", list(fingerprint.keys()))

In [ ]:
with open(os.path.join(preprocessed_dataset_dir, "nnUNetPlans.json")) as f:
    plans = json.load(f)

for config_name, config in plans["configurations"].items():
    print(f"--- {config_name} ---")
    print("  spacing:", config.get("spacing"))
    print("  patch_size:", config.get("patch_size"))
    print("  batch_size:", config.get("batch_size"))

::: {.callout-note}
Comparen estos valores de `spacing` y `patch_size` con los que usamos "a mano" en el `Notebook1` (`Spacingd(pixdim=(1.5, 1.5, 2.0))`, `RandCropByPosNegLabeld(spatial_size=(96, 96, 96))`). nnU-Net llegó a sus propios valores analizando el dataset, no copiándolos de un tutorial.
:::

## Paso 3 — Entrenar un único modelo

Entrenamos la configuración `3d_fullres`, fold 0, con el trainer reducido `nnUNetTrainer_5epochs` que definimos al crear el `runner`. En un caso real se entrenarían los 5 folds (para poder ensamblar y estimar variabilidad) y, muchas veces, más de una configuración.

::: {.callout-warning}
Aun con solo 5 épocas y un fold, este paso puede tardar varios minutos en una GPU de Colab. Si el tiempo es muy limitado, pueden probar con un `input_config` que use `trainer_class_name="nnUNetTrainer_1epoch"`.
:::

In [ ]:
runner.train_single_model(config="3d_fullres", fold=0)

### Visualizar el progreso del entrenamiento

nnU-Net guarda automáticamente un `progress.png` con las curvas de *loss* y *pseudo Dice* por época dentro de la carpeta de resultados de ese fold.

In [ ]:
fold0_dir = glob.glob(
    os.path.join(runner.nnunet_results, dataset_dir_name, "nnUNetTrainer_5epochs__nnUNetPlans__3d_fullres", "fold_0")
)[0]
progress_png = os.path.join(fold0_dir, "progress.png")

img = plt.imread(progress_png)
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis("off")
plt.title("Curvas de entrenamiento (loss / pseudo Dice) — fold 0")
plt.show()

## Paso 4 — Encontrar la mejor configuración

En un flujo completo, `find_best_configuration` compara todas las configuraciones/folds entrenados (incluyendo posibles ensambles) y elige la que mejor generaliza según la validación cruzada. Como nosotros solo entrenamos **una** configuración y **un** fold, este paso no tiene mucho para "elegir", pero corremos el comando igual para ver qué produce: un archivo `inference_information.json` que documenta la decisión (y que va a ser usado en el paso de predicción).

In [ ]:
runner.find_best_configuration(
    configs="3d_fullres",
    folds=(0,),
    allow_ensembling=False,
)

In [ ]:
inference_info_path = os.path.join(runner.nnunet_results, dataset_dir_name, "inference_information.json")
with open(inference_info_path) as f:
    inference_info = json.load(f)

print(json.dumps(inference_info["best_model_or_ensemble"], indent=2, ensure_ascii=False))

## Paso 5 — Predicción

Usamos `predict_ensemble_postprocessing` pero desactivando ensamble y postprocesamiento, para quedarnos solo con la **predicción cruda** del modelo sobre las imágenes de test (`imagesTs`).

In [ ]:
runner.predict_ensemble_postprocessing(
    folds=(0,),
    run_ensemble=False,
    run_predict=True,
    run_postprocessing=False,
)

### Visualizar una predicción

Tomamos el primer caso de test, y mostramos un corte axial de la imagen junto con la máscara predicha.

In [ ]:
pred_dir = os.path.join(runner.nnunet_results, dataset_dir_name, "pred_3d_fullres")
test_images_dir = os.path.join(runner.nnunet_raw, dataset_dir_name, "imagesTs")

pred_files = sorted(glob.glob(os.path.join(pred_dir, "*.nii.gz")))
case_id = os.path.basename(pred_files[0]).replace(".nii.gz", "")

image = nib.load(os.path.join(test_images_dir, f"{case_id}_0000.nii.gz")).get_fdata()
pred_mask = nib.load(pred_files[0]).get_fdata()

mid_slice = image.shape[2] // 2
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(image[:, :, mid_slice].T, cmap="gray")
axes[0].set_title(f"TC — caso {case_id}")
axes[0].axis("off")
axes[1].imshow(image[:, :, mid_slice].T, cmap="gray")
axes[1].imshow(pred_mask[:, :, mid_slice].T, cmap="Reds", alpha=0.4)
axes[1].set_title("Predicción (bazo) superpuesta")
axes[1].axis("off")
plt.show()

## Paso 6 — Postprocesamiento

El postprocesamiento que eligió `find_best_configuration` (guardado en `inference_information.json`) típicamente aplica reglas como "quedarse solo con la componente conexa más grande" para eliminar islas de falsos positivos aisladas. Corremos ahora solo esa etapa, sobre la predicción ya generada.

In [ ]:
runner.predict_ensemble_postprocessing(
    folds=(0,),
    run_predict=False,
    run_ensemble=False,
    run_postprocessing=True,
)

### Comparar la máscara antes y después del postprocesamiento

In [ ]:
postproc_dir = os.path.join(runner.nnunet_results, dataset_dir_name, "pred_3d_fullres_postprocessed")
postproc_files = sorted(glob.glob(os.path.join(postproc_dir, "*.nii.gz")))
postproc_mask = nib.load([p for p in postproc_files if case_id in p][0]).get_fdata()

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(image[:, :, mid_slice].T, cmap="gray")
axes[0].imshow(pred_mask[:, :, mid_slice].T, cmap="Reds", alpha=0.4)
axes[0].set_title("Antes del postprocesamiento")
axes[0].axis("off")
axes[1].imshow(image[:, :, mid_slice].T, cmap="gray")
axes[1].imshow(postproc_mask[:, :, mid_slice].T, cmap="Reds", alpha=0.4)
axes[1].set_title("Después del postprocesamiento")
axes[1].axis("off")
plt.show()

## Cierre: recorrido completo del pipeline

| Paso del notebook | Comando de `nnUNetV2Runner` | Concepto de la clase 3 |
|---|---|---|
| 1. Conversión | `convert_msd_dataset` | Preparar el `Dataset` de la figura |
| 2. Planning + preprocesamiento | `plan_and_process` | `Dataset fingerprint` → `Reglas de configuración` → `Preprocesamiento` |
| 3. Entrenamiento | `train_single_model` | `Configuración de red` → `Entrenamiento` |
| 4. Mejor configuración | `find_best_configuration` | Elegir entre configuraciones/folds/ensambles |
| 5. Predicción | `predict_ensemble_postprocessing(run_predict=True)` | `Inferencia` |
| 6. Postprocesamiento | `predict_ensemble_postprocessing(run_postprocessing=True)` | `Postprocesamiento` |

### Preguntas para reflexionar

1. Si este dataset tuviera **mucha anisotropía** (por ejemplo, cortes muy espaciados en el eje $z$), ¿qué esperarían que cambie en el `nnUNetPlans.json` (spacing, patch size, 2D vs 3D)?
2. ¿Por qué nnU-Net entrena **5 folds** por configuración en vez de uno solo? ¿Qué perdemos al entrenar un único fold como hicimos acá?
3. El postprocesamiento que vimos (componente conexa más grande) asume que hay **un solo bazo por paciente**. ¿En qué otras estructuras anatómicas esa misma regla sería inválida?
4. Retomando la pregunta central de la clase 3 ("¿reemplazar o colaborar?"): ¿en qué punto de este pipeline agregarían un paso de **revisión clínica** antes de aceptar la máscara final?